# generate_config() — test on 10 Low-Rise buildings

One call to `generate_config()` (in `generate_config.py`, same folder) goes from the inputs to all outputs:
generate the pool → evaluate it at the baseline context → select N diverse configurations (inside the function) → re-run **only the selected ones** under the
3 economic contexts → `summary_df` for the Monte-Carlo pipeline + figures.

REHO results are saved as they are produced: if the call is interrupted, calling it again resumes where it stopped.
`QUICK = True` uses a reduced lever set (about 16 + 16 + 3N REHO runs); set it to `False` for the full pool (it writes
to a new `OUT_DIR`).

In [ ]:
import importlib
import os
import sys

import pandas as pd

SCRIPTS_DIR = os.path.abspath('../..')                       # scripts/: uncertainty_postprocess.py
UNCERTAINTY_DIR = os.path.join(SCRIPTS_DIR, 'uncertainty')   # energy_storage_distributions.py + xlsx
sys.path[:0] = [SCRIPTS_DIR, UNCERTAINTY_DIR]

import energy_storage_distributions as esd
import generate_config as gc
import challenge_configurations as cc
for m in (gc, cc):                                            # pick up edits of the .py files without a restart
    importlib.reload(m)

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)

## Inputs

In [13]:
DISTRICT = 'LowRise'
QB_FILE = '../results/district_candidates_qbuildings.pickle'

# 10 Low-Rise buildings (drawn once with seed 2026, then fixed so every run uses the same ones)
BUILDINGS = ['1004675', '1004681', '1004687', '1004709', '1004710', '1004715', '1004724',
             '1004729/1004731', '2035402', '3131478/295075441/295092672']

WEATHER = {'Location': 'Geneva', 'Attributes': ['T', 'I', 'W'], 'Periods': 10, 'PeriodDuration': 24}
CONTEXTS = gc.DEFAULT_CONTEXTS                    # attractive / baseline / harsh (slide 2026-11-10)
UNCERTAIN_VAR_LINK = gc.DEFAULT_UNCERTAIN_VAR_LINK

QUICK = True
if QUICK:   # first test: few levers, to check that every stage runs
    LEVERS = {'TOTEX': {'lever': 'mga', 'eps': [0.1, 0.5], 'objectives': ['CAPEX_min', 'CAPEX_max', 'SS_max']},
              'SS': {'lever': 'ss_target', 'fractions': [0.5]},
              'price': {'lever': 'tariffs', 'pairs': [(0.40, 0.07)]},
              'installed_capacity': {'lever': 'sizes', 'rSOC_district': [6], 'Battery_district': [100],
                                     'roof_share': [0.5], 'tech_sets': ['no_storage', 'battery_only', 'rsoc_only']},
              'GU': None}
    OUT_DIR = f'results/{DISTRICT}10_gc_quick'
else:
    LEVERS = gc.DEFAULT_KPIS_OBJECTIVE_LINK
    OUT_DIR = f'results/{DISTRICT}10_gc'

N_MAX = 'auto'    # 'auto' | int | None

ARGS = dict(district_id=DISTRICT, uncertain_var_link=UNCERTAIN_VAR_LINK, KPIs_objective_link=LEVERS, n_max=N_MAX,
            contexts=CONTEXTS, out_dir=OUT_DIR, qb_file=QB_FILE, buildings=BUILDINGS, weather=WEATHER,
            postprocess_dir=SCRIPTS_DIR)
print(OUT_DIR)

results/LowRise10_gc_quick


Uncertainty distributions (supervisor's sheet). Used by step 2; `TECH_FROM_XLSX = True` also takes the design-stage
technology costs from their medians (`gc.DEFAULT_TECH` already holds the same values, so the default is `False`;
switching it needs a new `OUT_DIR`).

In [ ]:
ALIASES = {'normal': 'normal', 'log-normal': 'lognormal', 'lognormal': 'lognormal', 'log normal': 'lognormal',
           'uniform': 'uniform', 'triangular': 'triangular'}
DISTS = esd.build_distribution_dict(esd.load_parameters(os.path.join(UNCERTAINTY_DIR, 'energy_storage_parameters.xlsx'),
                                                        ALIASES), percentiles=(0.025, 0.975))

TECH_FROM_XLSX = False
if TECH_FROM_XLSX:
    ARGS['uncertainty_distributions'] = DISTS
print(gc.tech_from_distributions(DISTS) if TECH_FROM_XLSX else gc.DEFAULT_TECH)

## Run

In [15]:
out = gc.generate_config(**ARGS)

## Design codes

Every generated design is named `G<generator>-<nn>`; `nn` follows the order of the lever specification in `LEVERS`.

| Code | Generator | Designs (QUICK) |
|---|---|---|
| G0 | Anchors | G0-01 TOTEX optimum, G0-02 SS maximum |
| G1 | TOTEX optimum under other tariffs (RT / FI) | one per tariff pair |
| G2 | MGA: TOTEX ≤ (1 + ε) · TOTEX_opt, then min CAPEX / max CAPEX / max SS | for each ε: CAPEX_min, CAPEX_max, SS_max |
| G3 | Grid import fixed at a share of the way from the TOTEX optimum to the SS maximum | one per share |
| G4 | TOTEX optimum with a restricted storage set | no storage, battery only, rSOC only |
| G5 | TOTEX optimum with a forced minimum size or a reduced PV roof share | rSOC ≥ x, battery ≥ x, roof share |

The `description` column of the tables says exactly which run each code is.

**Clusters** = storage technology set actually installed in the design (size > 0): PV+HP (no storage), PV+HP+battery, PV+HP+rSOC, PV+HP+battery+rSOC. A design generated as "rSOC only" can therefore end up in *PV+HP (no storage)* if the optimizer installs no rSOC.

## Selected configurations by cluster

Only the N selected designs (the rejected pool designs are not kept in the outputs and are not run under the 3 contexts).
Left: cluster (technology class); then the design code; then what the design is, KPIs and installed capacities (evaluated at the baseline context).

In [16]:
out['cluster_table'].round(2)

description  TOTEX [CHF/m2/yr]  CAPEX [CHF/m2/yr]  OPEX [CHF/m2/yr]  SS [%]  GU [-]  Import [kWh/m2/yr]  Export [kWh/m2/yr]  PV [kW/1000m2]  HP [kW/1000m2]  Battery [kWh/1000m2]  rSOC [kW/1000m2]  \
cluster            design                                                                                                                                                                                                                               
PV+HP (no storage) G2-04                   eps 50%, CAPEX_min              25.42              15.53              9.89   36.17    3.11               62.73               89.25          102.39           34.87                  0.00              0.00   
                   G4-01                           no storage              17.88              21.94             -4.07   48.76    3.78               38.07              154.86          156.79           44.04                  0.00              0.00   
PV+HP+battery      G0-01                        TOTEX optimum              17.30              23.92             -6.63   67.63    3.78               24.52              139.85          156.79           44.04                 59.25              0.00   
                   G2-03                      eps 10%, SS_max              19.02              26.91             -7.89   75.22    3.33               18.60              134.64          156.79           54.37                143.14              0.00   
                   G5-03                    PV roof share 50%              22.93              18.29              4.64   52.35    3.62               34.65               57.49           78.40           45.43                 35.28              0.00   
PV+HP+battery+rSOC G0-02                           SS maximum              23.90              33.14             -9.24   99.91    3.25                0.09               92.64          156.79           59.09                 53.32             15.74   
                   G2-05                   eps 50%, CAPEX_max              25.66              35.50             -9.85   99.65    3.25                0.32               99.41          156.79           62.18                205.25             11.14   
                   G2-06                      eps 50%, SS_max              23.38              32.39             -9.01   99.94    3.45                0.06               90.27          156.79           52.92                 49.11             14.87   
                   G3-01   import at 50% of the way to SS max              20.27              28.12             -7.85   86.05    3.78               12.29              115.32          156.79           44.04                 66.64              4.07   
                   G5-01                 rSOC >= 6 kW/1000 m2              19.37              26.00             -6.63   67.64    3.78               24.52              139.85          156.79           44.04                 59.25              6.00   

                           CH4 storage [kWh/1000m2]  
cluster            design                            
PV+HP (no storage) G2-04                       0.00  
                   G4-01                       0.00  
PV+HP+battery      G0-01                       0.00  
                   G2-03                       0.00  
                   G5-03                       0.00  
PV+HP+battery+rSOC G0-02                   27091.18  
                   G2-05                   22636.13  
                   G2-06                   28216.41  
                   G3-01                   12418.45  
                   G5-01                       0.00

## Outputs

1. The 3 × N optimization results (pickles in `OUT_DIR`, one per design × context)

In [17]:
out['files']

RT    FI                                            file
design context                                                               
G0-01  attractive  0.20  0.15       LowRise_G0-01_battery_attractive_0.2_0.15
       baseline    0.30  0.10          LowRise_G0-01_battery_baseline_0.3_0.1
       harsh       0.04  0.02           LowRise_G0-01_battery_harsh_0.04_0.02
G4-01  attractive  0.20  0.15          LowRise_G4-01_none_attractive_0.2_0.15
       baseline    0.30  0.10             LowRise_G4-01_none_baseline_0.3_0.1
       harsh       0.04  0.02              LowRise_G4-01_none_harsh_0.04_0.02
G5-01  attractive  0.20  0.15  LowRise_G5-01_battery+rsoc_attractive_0.2_0.15
       baseline    0.30  0.10     LowRise_G5-01_battery+rsoc_baseline_0.3_0.1
       harsh       0.04  0.02      LowRise_G5-01_battery+rsoc_harsh_0.04_0.02
G2-05  attractive  0.20  0.15  LowRise_G2-05_battery+rsoc_attractive_0.2_0.15
       baseline    0.30  0.10     LowRise_G2-05_battery+rsoc_baseline_0.3_0.1
       harsh       0.04  0.02      LowRise_G2-05_battery+rsoc_harsh_0.04_0.02
G2-06  attractive  0.20  0.15  LowRise_G2-06_battery+rsoc_attractive_0.2_0.15
       baseline    0.30  0.10     LowRise_G2-06_battery+rsoc_baseline_0.3_0.1
       harsh       0.04  0.02      LowRise_G2-06_battery+rsoc_harsh_0.04_0.02
G2-04  attractive  0.20  0.15          LowRise_G2-04_none_attractive_0.2_0.15
       baseline    0.30  0.10             LowRise_G2-04_none_baseline_0.3_0.1
       harsh       0.04  0.02              LowRise_G2-04_none_harsh_0.04_0.02
G2-03  attractive  0.20  0.15       LowRise_G2-03_battery_attractive_0.2_0.15
       baseline    0.30  0.10          LowRise_G2-03_battery_baseline_0.3_0.1
       harsh       0.04  0.02           LowRise_G2-03_battery_harsh_0.04_0.02
G3-01  attractive  0.20  0.15  LowRise_G3-01_battery+rsoc_attractive_0.2_0.15
       baseline    0.30  0.10     LowRise_G3-01_battery+rsoc_baseline_0.3_0.1
       harsh       0.04  0.02      LowRise_G3-01_battery+rsoc_harsh_0.04_0.02
G5-03  attractive  0.20  0.15       LowRise_G5-03_battery_attractive_0.2_0.15
       baseline    0.30  0.10          LowRise_G5-03_battery_baseline_0.3_0.1
       harsh       0.04  0.02           LowRise_G5-03_battery_harsh_0.04_0.02
G0-02  attractive  0.20  0.15  LowRise_G0-02_battery+rsoc_attractive_0.2_0.15
       baseline    0.30  0.10     LowRise_G0-02_battery+rsoc_baseline_0.3_0.1
       harsh       0.04  0.02      LowRise_G0-02_battery+rsoc_harsh_0.04_0.02

2. Dataframe for the Monte-Carlo simulation (design first, then context) and which column each uncertain variable acts on

In [18]:
sd = out['summary_df']
print(sd.shape)
print(out['uncertain_var_columns'])
sd[['techno', 'retail_elec', 'feed_in_elec', 'Elec_import', 'Elec_export', 'Battery_daily_size', 'rSOC_size',
    'CH4_storage_size', 'TOTEX_kpi', 'CAPEX_kpi', 'SS_kpi', 'GU_kpi']].round(2)

(30, 90)
{'electricity_feed_in': 'el_export_kpi', 'electricity_retail': 'el_import_kpi', 'battery_capex': 'Battery_district_kpi', 'rsoc_capex': 'rSOC_district_kpi', 'rsoc_lifetime': 'rSOC_district_kpi', 'ch4_tank_capex': 'CH4_storage_IP_district_kpi'}


techno  retail_elec  feed_in_elec  Elec_import  Elec_export  Battery_daily_size  rSOC_size  CH4_storage_size  TOTEX_kpi  CAPEX_kpi  SS_kpi  GU_kpi
design context                                                                                                                                                             
G0-01  attractive       battery         0.20          0.15       100.79       539.36              219.31       0.00              0.00       7.51      23.92   62.50    3.03
       baseline         battery         0.30          0.10        90.78       517.66              219.31       0.00              0.00      17.30      23.92   67.63    3.78
       harsh            battery         0.04          0.02        91.57       524.18              219.31       0.00              0.00      22.08      23.92   66.67    3.86
G4-01  attractive          none         0.20          0.15       155.41       599.90                0.00       0.00              0.00       6.02      21.94   40.88    3.03
       baseline            none         0.30          0.10       140.92       573.26                0.00       0.00              0.00      17.88      21.94   48.76    3.78
       harsh               none         0.04          0.02       145.14       583.58                0.00       0.00              0.00      20.35      21.94   46.03    3.86
G5-01  attractive  battery+rsoc         0.20          0.15       100.77       539.36              219.31      22.21              0.00       9.58      26.00   62.51    3.03
       baseline    battery+rsoc         0.30          0.10        90.76       517.66              219.31      22.21              0.00      19.37      26.00   67.64    3.78
       harsh       battery+rsoc         0.04          0.02        91.55       524.17              219.31      22.21              0.00      24.15      26.00   66.68    3.86
G2-05  attractive  battery+rsoc         0.20          0.15        62.27       505.21              759.75      41.23          83791.05      18.40      35.50   76.45    3.42
       baseline    battery+rsoc         0.30          0.10         1.20       367.99              759.75      41.23          83791.05      25.66      35.50   99.65    3.25
       harsh       battery+rsoc         0.04          0.02        63.58       500.94              759.75      41.23          83791.05      33.48      35.50   76.45    3.33
G2-06  attractive  battery+rsoc         0.20          0.15       107.39       550.57              181.80      55.04         104447.27      15.89      32.39   59.35    3.31
       baseline    battery+rsoc         0.30          0.10         0.21       334.15              181.80      55.04         104447.27      23.38      32.39   99.94    3.45
       harsh       battery+rsoc         0.04          0.02        96.87       533.31              181.80      55.04         104447.27      30.56      32.39   64.25    3.32
G2-04  attractive          none         0.20          0.15       239.67       347.39                0.00       0.00              0.00      14.40      15.53   32.34    2.79
       baseline            none         0.30          0.10       232.20       330.36                0.00       0.00              0.00      25.42      15.53   36.17    3.11
       harsh               none         0.04          0.02       233.10       336.02                0.00       0.00              0.00      16.23      15.53   35.08    3.15
G2-03  attractive       battery         0.20          0.15        68.48       509.20              529.85       0.00              0.00       9.97      26.91   74.32    3.25
       baseline         battery         0.30          0.10        68.86       498.39              529.85       0.00              0.00      19.02      26.91   75.22    3.33
       harsh            battery         0.04          0.02        67.54       502.83              529.85       0.00              0.00      24.92      26.91   75.18    3.26
G3-01  attractive  battery+rsoc         0.20          0.15        95.

3. Selection report: coverage of every KPI / capacity, as a share of the pool range

In [19]:
sel = out['selection']
print('selected', len(sel['selected']), 'of', sel['n_pool'], 'generated | auto N =', sel['n_auto'], '| rule met:', sel['rule_met'])
print('near-duplicates dropped:', sel['dropped_duplicates'])
sel['report'].round(2)

selected 10 of 16 generated | auto N = 10 | rule met: True
near-duplicates dropped: {'G4-02': 'G0-01', 'G1-01': 'G0-01', 'G4-03': 'G4-01', 'G2-01': 'G4-01'}


,range_cov_%,bands,bands_ok,KS
feature,,,,
TOTEX,100.0,4/4,True,0.12
CAPEX,100.0,4/4,True,0.07
SS,100.0,4/4,True,0.08
GU,100.0,4/4,True,0.08
PV,100.0,3/3,True,0.03
HeatPump_Air,100.0,4/4,True,0.05
Battery_district,100.0,4/4,True,0.13
rSOC_district,100.0,4/4,True,0.08
CH4_storage_IP_district,100.0,3/3,True,0.07


4. Figures (also saved as HTML in `OUT_DIR/figures`); colour = technology class.

- `kpis_vs_capex`: TOTEX, SS and GU (y) against CAPEX (x) for every selected design × context (symbol = context).
- `indicator_ranges`: range of SS, GU, TOTEX, CAPEX and of every installed capacity per technology class (baseline context; capacities are fixed by the design).
- `totex_bars`: all selected designs in one figure, REHO cost template (CAPEX left / OPEX right / thin TOTEX), grouped by technology class (shaded band), sorted by CAPEX inside each class, baseline context; comfort penalty shown explicitly.
- `uncertain_var_link`: one bar subplot per entry of `UNCERTAIN_VAR_LINK` (uncertain variable → what it acts on), designs grouped by technology class; operation results (import, export) one bar per context (pattern), installed capacities one bar (baseline context).

In [20]:
for name, fig in out['figures'].items():
    fig.show()

## Step 2: challenge_configurations_under_uncertainty()

Inputs: the uncertainty distributions (supervisor's xlsx) and the configuration dataframe `out['summary_df']` (N designs × 3 operations).
Paired Monte Carlo (same draws for every design, supervisor's `uncertainty_postprocess.py`), all costs in CHF/m²/yr. Per draw:

- **cost** of a design = TOTEX of the operation with the best score (TOTEX, SS, GU) in that draw (`operation='best'`) or the mean of its 3 operations (`operation='blend'`)
- **adaptability** = TOTEX with the baseline operation − TOTEX with the cheapest operation (what re-optimising the operation recovers)
- **lock-in** = TOTEX with the cheapest operation − TOTEX of the cheapest design (with its cheapest operation) (what stays lost even with the best operation)
- **regret** = cost − TOTEX of the cheapest design (with its cheapest operation)
- **score** (TOTEX, SS, GU): in every draw each KPI is scaled over the designs to 0 (best) .. 1 (worst), score = weighted sum (`kpi_weights`, default 1/3 each); SS and GU are those of the operation used in that draw
- **ranking**: rank of the score in each draw → probability of each rank, P(best), expected rank
- **blend (main)**: weight of a design = share of the sampled contexts in which it has the best score (P(best)); `ch['weights']`, TOTEX, SS and GU of the blend per draw in `ch['per_draw']['blend']`
- **blend (supplement)**: risk-averse weights minimising (1 − λ)·mean + λ·CVaR(`alpha`) of the score regret; `ch['frontier']` (every λ), `ch['weights_cvar']` (λ = 1); numbers only, no figure

Figures: `capex_totex` (median and 95 % interval of each axis), `capex_totex_cloud` (all draws + region holding 95 % of them), `ranking` (blend weights | TOTEX, SS, GU of every design vs the blend; rank probabilities in `ch['rank_probability']`), `regret` (adaptability vs lock-in).

In [21]:
CH_ARGS = dict(n_runs=1000, operation='best', initial_operation='baseline',
               kpi_weights={'TOTEX': 1/3, 'SS': 1/3, 'GU': 1/3}, alpha=0.9, out_dir=OUT_DIR)

ch = cc.challenge_configurations_under_uncertainty(DISTS, out['summary_df'], **CH_ARGS)
print(ch['low_regret'])
ch['summary'].round(2)

{'min_mean_regret': 'G0-01', 'min_p95_regret': 'G0-01', 'kpi_weights': {'TOTEX': 0.333, 'SS': 0.333, 'GU': 0.333}, 'blend': {'G0-01': 0.701, 'G0-02': 0.297}, 'blend_TOTEX_median': 23.330814957859577, 'blend_SS_median': 73.66276893202703, 'blend_GU_median': 3.098189415349077, 'blend_regret_mean': 2.2640036742068377, 'blend_regret_p95': 4.132582880032883, 'blend_cvar': {'G0-01': 0.521, 'G0-02': 0.479}}


,tech_set,CAPEX_median,CAPEX_lo95,CAPEX_hi95,TOTEX_median,TOTEX_lo95,TOTEX_hi95,SS_median,GU_median,score_mean,score_regret_mean,score_regret_cvar,P_best,expected_rank,weight,weight_cvar,regret_mean,regret_p95,adaptability_mean,adaptability_p95,lock_in_mean,lock_in_p95
design,,,,,,,,,,,,,,,,,,,,,,
G0-01,PV+HP+battery,23.92,23.48,24.34,21.66,15.29,30.09,62.50,3.03,0.28,0.02,0.14,0.7,1.80,0.7,0.52,0.73,2.00,0.06,0.12,0.27,1.38
G2-03,PV+HP+battery,26.91,25.85,27.91,22.76,17.22,29.02,75.18,3.26,0.32,0.06,0.13,0.0,3.02,0.0,0.00,1.66,3.51,0.24,0.32,1.64,3.51
G0-02,PV+HP+battery+rSOC,33.19,31.44,36.97,26.91,23.13,31.25,99.91,3.25,0.32,0.06,0.16,0.3,3.43,0.3,0.48,5.87,11.23,0.04,0.18,5.82,10.92
G5-01,PV+HP+battery+rSOC,26.03,25.16,27.70,23.92,17.41,32.46,62.51,3.03,0.34,0.09,0.18,0.0,4.05,0.0,0.00,2.95,4.64,0.06,0.12,2.49,4.08
G2-06,PV+HP+battery+rSOC,32.44,30.77,36.07,26.31,22.65,30.48,99.94,3.45,0.36,0.11,0.20,0.0,5.76,0.0,0.00,5.26,10.67,0.04,0.19,5.22,10.21
G3-01,PV+HP+battery+rSOC,28.17,27.31,29.53,25.64,19.50,33.75,64.49,3.03,0.39,0.13,0.21,0.0,6.31,0.0,0.00,4.69,6.13,0.03,0.19,2.64,5.39
G2-05,PV+HP+battery+rSOC,35.63,33.46,38.71,28.93,24.72,32.60,99.65,3.25,0.37,0.12,0.19,0.0,6.44,0.0,0.00,7.71,12.89,0.05,0.36,7.66,12.46
G4-01,PV+HP (no storage),21.94,21.94,21.94,22.83,14.19,35.15,40.88,3.03,0.42,0.17,0.34,0.0,6.86,0.0,0.00,2.06,6.74,0.01,0.04,1.38,5.16
G5-03,PV+HP+battery,18.29,18.03,18.54,24.49,18.32,34.80,48.37,3.08,0.45,0.20,0.33,0.0,8.19,0.0,0.00,3.90,7.31,0.00,0.02,3.56,6.69


In [22]:
for name, fig in ch['figures'].items():
    fig.show()